# Modified mpgmm Python modules for LAMIA

Much faster than the Matlab versions. Get global T-matrix

Input:
* Individual scatterer T-matrices in separate files
* Positions of the scatterers
* Multipolar order
* Wavelengths and backgrounds can be set directly

Output: Global T-matrix

In [7]:
import scipy.io as io
from tqdm.notebook import trange
import functions_vswf_tmatclusters
from sta_hel2em import sta_hel2em
import numpy as np

data_spheres = io.loadmat("xavi_example_pois/l12p5_ff40_v2.mat")
data_nbg = io.loadmat("xavi_example_pois/eps_matrices.mat")
nbg = np.sqrt(data_nbg['eps_al2o3'])
wl = data_nbg['wl']

n_max_local = 3

n_max_local_shape = functions_vswf_tmatclusters.make_mn(n_max_local)
T_global_shape = n_max_local_shape.shape[0]*n_max_local_shape.shape[1
T_global = np.zeros((wl.shape[1],T_global_shape,T_global_shape),dtype=complex)

for i_wavelength in trange(wl.shape[1], desc="Wavelengths loop"):
    wavelength = wl[0][i_wavelength]
    n_r = nbg[0][i_wavelength]
    k_mod = 2*np.pi*n_r/wavelength
    
    origin = np.array([0,0,0])
    positions = np.c_[data_spheres['x'],data_spheres['y'],data_spheres['z']]
    pos2orig = origin-positions

    T_mat_append = []

    for i_sphere in range(data_spheres['r'].shape[0]):
        i_sphere_ = i_sphere + 1
        filename_data_T = "xavi_example_pois/T_n3_%s.mat" % i_sphere_
        data_T = io.loadmat(filename_data_T)
        T_1 = data_T['T'][i_wavelength]
        T_mat_append.append(T_1)
    T_mat_append = np.array(T_mat_append)   
    T_global[i_wavelength,:,:] = functions_vswf_tmatclusters.Tmat_global(T_mat_append,n_max_local,positions,k_mod)

# if helicity basis needed
T_global_hel = sta_hel2em(T_global)